# M5 — Feature Engineering

Builds the final feature matrix for model training. All feature-construction
logic lives in [`src/features/build_features.py`](../src/features/build_features.py) —
this notebook just calls it and inspects/sanity-checks the result.

Inputs (`data/interim/`):
- `training_dataset.csv` — NKC + Goodreads + SCKN labels
- `pre_cutoff_stats.csv` — pre-cutoff review aggregates (no leakage)
- `goodreads_author_names.json` — author_id → name (for reference)

Outputs (`data/processed/`):
- `training_features.csv` — full feature matrix with label column retained
- `X_train.csv` — feature columns only
- `y_train.csv` — label column only

## Known limitations

- **Goodreads snapshot is from 2017.** Books with `czech_pub_year > 2017` have systematically lower pre-cutoff signal than they would in reality, because Goodreads reviews added between 2017 and the actual Czech publication date are missing from the dataset.
- **Overall NKC → Goodreads match rate is ~25%.** The remaining records have no Goodreads features and are excluded from training; model predictions are implicitly conditioned on a book being findable in the Goodreads catalogue.
- **Pre-cutoff signal is sparse.** A large share of training records have zero pre-cutoff ratings. Features derived from pre-cutoff reviews are unreliable for many books; `has_precutoff_signal` flags records with ≥ 5 ratings.


In [1]:
import sys
from pathlib import Path

REPO = Path("__file__").resolve().parents[1] if "__file__" in dir() else Path.cwd().parent
sys.path.insert(0, str(REPO))

from src.features.build_features import (
    load_inputs,
    merge_precutoff,
    build_feature_matrix,
    feature_cols,
    save_outputs,
)

print(f"REPO : {REPO}")

REPO : /home/firstone/Bachelors-thesis


## Step 1 — Load and merge

In [2]:
train, pre, author_names = load_inputs()
print(f"training_dataset : {len(train):,} rows, {train.columns.tolist()}")
print(f"pre_cutoff_stats  : {len(pre):,} rows, {pre.columns.tolist()}")
print(f"author_names      : {len(author_names):,} entries")

training_dataset : 24,161 rows, ['nkc_id', 'oclc', 'czech_isbn', 'czech_title', 'original_title', 'original_isbn', 'original_sysnum', 'author', 'secondary_authors', 'czech_pub_year', 'first_czech_year', 'source_lang', 'genres', 'match_layer', 'matched_book_id', 'gr_work_id', 'fuzzy_score', 'gr_title', 'gr_pub_year', 'gr_ratings_count', 'gr_average_rating', 'gr_text_reviews_count', 'gr_popular_shelves', 'gr_language_code', 'gr_is_ebook', 'sckn_appearances', 'sckn_first_year', 'sckn_best_rank', 'sckn_categories', 'sckn_bestseller']
pre_cutoff_stats  : 23,670 rows, ['matched_book_id', 'czech_pub_year', 'pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'pre_cutoff_reviews_with_text']
author_names      : 829,524 entries


In [3]:
df = merge_precutoff(train, pre)
print(f"After merge: {len(df):,} rows")
print(f"  Books with zero pre-cutoff ratings: {(df['pre_cutoff_ratings_count'] == 0).sum():,}")

After merge: 24,161 rows
  Books with zero pre-cutoff ratings: 8,622


## Step 2 — Build feature matrix

In [4]:
feat = build_feature_matrix(df)

print("Feature matrix shape:", feat.shape)
print("Columns:", feat.columns.tolist())

Feature matrix shape: (24161, 18)
Columns: ['pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'log_pre_cutoff_ratings_count', 'has_precutoff_signal', 'gr_ratings_count', 'shelf_fiction', 'shelf_mystery', 'shelf_romance', 'shelf_scifi', 'shelf_nonfiction', 'shelf_ya', 'shelf_classics', 'lang_eng', 'lang_ger', 'lang_fre', 'lang_rus', 'czech_pub_year', 'sckn_bestseller']


In [5]:
print("Popularity features:")
print(feat[["pre_cutoff_ratings_count", "pre_cutoff_avg_rating",
            "log_pre_cutoff_ratings_count", "has_precutoff_signal",
            "gr_ratings_count"]].describe())

Popularity features:
       pre_cutoff_ratings_count  pre_cutoff_avg_rating  \
count              24161.000000           24161.000000   
mean                  71.016100               3.735574   
std                  312.545068               0.521686   
min                    0.000000               1.000000   
25%                    0.000000               3.500000   
50%                    3.000000               3.500000   
75%                   24.000000               4.000000   
max                13008.000000               5.000000   

       log_pre_cutoff_ratings_count  has_precutoff_signal  gr_ratings_count  
count                  24161.000000          24161.000000      2.416100e+04  
mean                       1.881367              0.441124      3.060632e+03  
std                        1.996549              0.496532      2.655454e+04  
min                        0.000000              0.000000      1.000000e+01  
25%                        0.000000              0.000000      4.8

In [6]:
shelf_cols = [c for c in feat.columns if c.startswith("shelf_")]
print("Shelf bucket distributions:")
print(feat[shelf_cols].describe().loc[["mean", "max"]].round(4))

Shelf bucket distributions:
      shelf_fiction  shelf_mystery  shelf_romance  shelf_scifi  \
mean         0.0166         0.0134         0.0079       0.0151   
max          0.2921         0.4000         0.3534       0.5939   

      shelf_nonfiction  shelf_ya  shelf_classics  
mean            0.0195    0.0079          0.0009  
max             0.4641    0.2805          0.2842  


In [7]:
lang_cols = [c for c in feat.columns if c.startswith("lang_")]
print("Source language counts:")
print(feat[lang_cols].sum().to_string())
print(f"\nRecords covered by the language dummies: {feat[lang_cols].any(axis=1).sum():,} / {len(feat):,}")

Source language counts:
lang_eng    20633
lang_ger      811
lang_fre      829
lang_rus        5

Records covered by the language dummies: 22,278 / 24,161


In [8]:
print("Year range:", feat["czech_pub_year"].min(), "–", feat["czech_pub_year"].max())
print("Year distribution:")
print(feat["czech_pub_year"].describe())

Year range: 2003 – 2026
Year distribution:
count        24161.0
mean     2012.818344
std         5.540682
min           2003.0
25%           2008.0
50%           2013.0
75%           2017.0
max           2026.0
Name: czech_pub_year, dtype: Float64


## Step 3 — Sanity checks

In [9]:
f_cols = feature_cols(feat)
n_rows = len(feat)
print(f"Rows          : {n_rows:,}")
print(f"Feature cols  : {len(f_cols)}: {f_cols}")
print(f"Label col     : sckn_bestseller")
print(f"Metadata cols : gr_ratings_count (training_features.csv only)")

Rows          : 24,161
Feature cols  : 16: ['pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'log_pre_cutoff_ratings_count', 'has_precutoff_signal', 'shelf_fiction', 'shelf_mystery', 'shelf_romance', 'shelf_scifi', 'shelf_nonfiction', 'shelf_ya', 'shelf_classics', 'lang_eng', 'lang_ger', 'lang_fre', 'lang_rus', 'czech_pub_year']
Label col     : sckn_bestseller
Metadata cols : gr_ratings_count (training_features.csv only)


In [10]:
import pandas as pd

missing = feat[f_cols].isnull().sum()
missing_pct = missing / n_rows * 100
missing_report = pd.DataFrame({"missing": missing, "pct": missing_pct.round(2)})
print("Missing values per column:")
print(missing_report[missing_report["missing"] > 0].to_string())

for col in f_cols:
    pct = feat[col].isnull().mean() * 100
    if pct > 5:
        print(f"  WARNING: {col} has {pct:.1f}% missing — consider imputation or dropping")

Missing values per column:
Empty DataFrame
Columns: [missing, pct]
Index: []


In [11]:
n_pos = feat["sckn_bestseller"].sum()
n_neg = n_rows - n_pos
print(f"Class distribution:")
print(f"  Bestseller (1): {n_pos:>6,}  ({n_pos/n_rows:.1%})")
print(f"  Non-best.  (0): {n_neg:>6,}  ({n_neg/n_rows:.1%})")
print(f"  Imbalance ratio: {n_neg/n_pos:.1f}:1")

Class distribution:
  Bestseller (1):  1,287  (5.3%)
  Non-best.  (0): 22,874  (94.7%)
  Imbalance ratio: 17.8:1


In [12]:
numeric_feats = feat[f_cols].select_dtypes(include="number").columns.tolist()
corrs = feat[numeric_feats + ["sckn_bestseller"]].corr()["sckn_bestseller"].drop("sckn_bestseller")
top10 = corrs.abs().sort_values(ascending=False).head(10)

print("Top 10 features by |correlation| with sckn_bestseller:")
print(corrs[top10.index].round(4).to_string())

Top 10 features by |correlation| with sckn_bestseller:
pre_cutoff_ratings_count        0.1476
shelf_ya                        0.0875
log_pre_cutoff_ratings_count    0.0772
shelf_fiction                   0.0635
shelf_mystery                   0.0509
czech_pub_year                 -0.0392
has_precutoff_signal            0.0343
lang_fre                        0.0241
lang_eng                       -0.0235
shelf_nonfiction               -0.0129


## Step 4 — Save outputs

In [13]:
save_outputs(feat)

X = feat[f_cols]
y = feat[["sckn_bestseller"]]
print(f"Saved {len(feat):,} rows -> training_features.csv")
print(f"Saved X_train.csv  shape={X.shape}")
print(f"X_train columns ({len(X.columns)}): {X.columns.tolist()}")
print(f"Saved y_train.csv  shape={y.shape}")

Saved 24,161 rows -> training_features.csv
Saved X_train.csv  shape=(24161, 16)
X_train columns (16): ['pre_cutoff_ratings_count', 'pre_cutoff_avg_rating', 'log_pre_cutoff_ratings_count', 'has_precutoff_signal', 'shelf_fiction', 'shelf_mystery', 'shelf_romance', 'shelf_scifi', 'shelf_nonfiction', 'shelf_ya', 'shelf_classics', 'lang_eng', 'lang_ger', 'lang_fre', 'lang_rus', 'czech_pub_year']
Saved y_train.csv  shape=(24161, 1)
